# LEXIS Master E2E Validation Notebook
This notebook serves as the master production testing suite for the LEXIS system. As we add new architectural phases, we will add cells here to test the entire E2E flow from ingestion to retrieval to serving in a completely unmocked environment.

## 1. Environment & Infrastructure Setup
Install dependencies and start background services (Redis, Elasticsearch, etc) native to the Colab container.

In [ ]:
!apt-get update -qq
!apt-get install -y redis-server -qq
!redis-server --daemonize yes

!pip install -q redis pydantic fastapi httpx litellm qdrant-client sentence-transformers groq tiktoken

## 2. Configuration
Set your API keys and verify the local Redis cluster is running.

In [ ]:
import os
import redis

os.environ['GROQ_API_KEY'] = 'gsk_your_groq_key' # Replace with actual key to run
os.environ['LLM_MODEL'] = 'groq/llama-3.3-70b-versatile'
os.environ['REDIS_URL'] = 'redis://localhost:6379/0'

r = redis.Redis(host='localhost', port=6379, db=0)
if r.ping():
    print('✅ Live Redis Server is UP and responding.')
else:
    print('❌ Redis failed to start.')

## 3. Clone Repository & Setup Path
*Uncomment the git clone command if running outside your local repository context.*

In [ ]:
# !git clone https://github.com/your-username/RAG-Basic.git
import sys
sys.path.append('/content/RAG-Basic/LEXIS/src') # Adjust path as necessary for colab

## 4. Initialization (Ingestion Phase)
Initialize the Qdrant collections and insert test documents to populate the 4 retrieval paths (Primary, HYPE, Propositions, RAPTOR).

In [ ]:
import asyncio
from qdrant_client.http.models import PointStruct
from lexis.retrieval.engine import RetrievalEngine

engine = RetrievalEngine()

async def seed_qdrant():
    await engine.qdrant.initialize_collections()
    # Insert base chunks for provenance validation
    for col in ['primary_v2', 'hype_v2', 'propositions_v2', 'clusters_v2']:
        await engine.qdrant.upsert_chunks(col, [
            PointStruct(
                id=1, 
                vector=[0.1]*384, 
                payload={
                    'chunk_id': f'{col}_1', 
                    'content': 'Termination for cause requires 30 days notice of un-remedied breach.', 
                    'doc_id': 'doc_1', 
                    'pqac_key': 'pqac-12345678-1234-1234-1234-123456789abc'
                }
            )
        ])
    print('✅ Qdrant Initialized & Seeded')

await seed_qdrant()

## 5. Start Deep Mode Worker (Serving Phase)
Start the Redis stream consumer group. This simulates the decoupled background worker in production.

In [ ]:
from lexis.serving.redis_manager import RedisManager
from lexis.serving.worker import DeepModeWorker

redis_manager = RedisManager()
worker = DeepModeWorker(redis_manager)
worker.engine = engine # Inject the seeded engine

worker_task = asyncio.create_task(worker.run())
print('✅ Background Worker Listening on Redis Streams...')

## 6. End-to-End Simulation (FastAPI -> Worker -> SSE)
Act as the frontend to enqueue a query and stream the response via PubSub Server-Sent Events.

In [ ]:
import json
from lexis.serving.api import router, query_deep_events
from fastapi.testclient import TestClient

client = TestClient(router)

async def run_e2e():
    print('[Test] 1. Enqueueing Deep Mode Job via FastAPI...')
    response = client.post(
        '/v2/query/deep', 
        json={'query': 'What is the notice period for termination for cause?', 'metadata_filters': {}},
        headers={'X-API-Key': 'test_tenant'}
    )
    
    assert response.status_code == 200
    job_id = response.json().get('job_id')
    print(f'✅ Job Enqueued: {job_id}')

    print('[Test] 2. Subscribing to SSE Event Stream...')
    class MockRequest:
        async def is_disconnected(self): return False
        
    generator = await query_deep_events(job_id, MockRequest())
    
    state_sequence = []
    tokens = []
    
    print('\n--- LIVE SSE EVENT STREAM ---')
    async for raw_event in generator.body_iterator:
        event_str = raw_event.strip()
        if not event_str: continue
            
        lines = event_str.split('\n')
        event_type = lines[0].replace('event: ', '').strip()
        data_json = json.loads(lines[1].replace('data: ', '').strip())
        
        if event_type == 'status':
            state = data_json.get('state')
            state_sequence.append(state)
            print(f'🟡 STATE CHANGE: {state}')
            
        elif event_type == 'progress':
            tokens.append(data_json.get('token'))
            print(data_json.get('token'), end='', flush=True)
            
        elif event_type in ['completed', 'failed']:
            state = data_json.get('state')
            state_sequence.append(state)
            print(f'\n🟢 TERMINAL EVENT: {state}')
            break
            
    print('\n-----------------------------\n')
    
    expected = ['RUNNING', 'MAP_PHASE', 'REDUCE_PHASE', 'SYNTHESIS', 'COMPLETED']
    if expected == state_sequence:
        print('✅ Exact State Machine Validated')
    else:
        print(f'❌ State Machine Mismatch! Got: {state_sequence}')
        
    if tokens:
        print('✅ Tokens Streamed Validated')
    else:
        print('❌ No Tokens Streamed')

await run_e2e()
worker_task.cancel()